### ============================================================
### Assignment 2 — Instructor Version (Sample Answers)
### Validation, Overfitting, and Forecast Failure
### ECON 417 · Business Forecasting · SIUE
### ============================================================


> **⚠️ Instructor Version** — This notebook contains sample answers for all written questions and complete code for all coding questions. Do not distribute to students.


---
## Setup — run this cell first, do not modify

In [ ]:
# ── Setup — do not modify ────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

COLORS = {
    'train'  : '#2196F3',
    'val'    : '#FF9800',
    'test'   : '#4CAF50',
    'neutral': '#546E7A',
    'accent' : '#F4A261',
    'red'    : '#E53935',
    'purple' : '#7B1FA2',
}

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# ── Synthetic S&P 500 ecosystem ──────────────────────────────────────────
DATA_SOURCE = 'synthetic'
try:
    import yfinance as yf
    _raw = yf.download(['^GSPC','^VIX','MSFT','AAPL','JPM','XOM'],
                       start='2015-01-01', end='2024-12-31',
                       auto_adjust=True, progress=False)
    if len(_raw) > 100:
        close  = _raw['Close'].dropna()
        sp500  = close['^GSPC']
        vix    = _raw['Close']['^VIX'].reindex(sp500.index).ffill()
        msft   = close['MSFT'];  aapl = close['AAPL']
        jpm    = close['JPM'];   xom  = close['XOM']
        DATA_SOURCE = 'real (yfinance)'
    else:
        raise ValueError
except Exception:
    np.random.seed(42)
    dates  = pd.date_range('2015-01-02', '2024-12-31', freq='B')
    N      = len(dates)
    sp_ret = np.random.normal(0.0003, 0.010, N)
    sp500  = pd.Series(4000 * np.exp(np.cumsum(sp_ret)), index=dates)
    vix    = pd.Series(np.clip(
                 18 + np.cumsum(np.random.normal(0, 0.2, N) - 0.3*sp_ret*10),
                 10, 80), index=dates)
    def mk(base, beta, iv):
        r = beta*sp_ret + np.random.normal(0, iv, N)
        return pd.Series(base*np.exp(np.cumsum(r)), index=dates)
    msft = mk(300,1.2,0.012); aapl = mk(150,1.15,0.013)
    jpm  = mk( 80,1.05,0.011); xom = mk( 60,0.80,0.014)

df   = pd.DataFrame({'SP500':sp500,'VIX':vix,'MSFT':msft,
                     'AAPL':aapl,'JPM':jpm,'XOM':xom}).dropna()
rets = np.log(df[['SP500','MSFT','AAPL','JPM','XOM']]).diff().dropna()
rets['VIX_chg'] = df['VIX'].diff().reindex(rets.index)
rets = rets.dropna()

# ── Target: S&P 500 log returns ──────────────────────────────────────────
log_ret = rets['SP500'].values
dates_r = rets.index
n       = len(log_ret)

# ── Chronological 60 / 20 / 20 split ─────────────────────────────────────
n_tr  = int(n * 0.60)
n_val = int(n * 0.20)
n_te  = n - n_tr - n_val

idx_tr  = slice(0, n_tr)
idx_val = slice(n_tr, n_tr + n_val)
idx_te  = slice(n_tr + n_val, n)

dates_tr  = dates_r[idx_tr]
dates_val = dates_r[idx_val]
dates_te  = dates_r[idx_te]

print(f'Data source : {DATA_SOURCE}')
print(f'Total obs   : {n}')
print(f'Train  : {n_tr:4d} obs  ({dates_tr[0].date()} → {dates_tr[-1].date()})')
print(f'Val    : {n_val:4d} obs  ({dates_val[0].date()} → {dates_val[-1].date()})')
print(f'Test   : {n_te:4d} obs  ({dates_te[0].date()} → {dates_te[-1].date()})')
print('\nTest set is LOCKED — do not use it until Q7.')


### Data Split Overview — run this cell to see your train/val/test periods

In [ ]:
# ── Pre-built: visualise the 60/20/20 chronological split ───────────────
fig, ax = plt.subplots(figsize=(13, 3))
sp_full = np.log(df['SP500']).diff().dropna().reindex(dates_r)
ax.fill_between(dates_tr,  sp_full.iloc[idx_tr],  alpha=0.35, color=COLORS['train'], label=f'Train  ({n_tr} obs)')
ax.fill_between(dates_val, sp_full.iloc[idx_val], alpha=0.45, color=COLORS['val'],   label=f'Val    ({n_val} obs)')
ax.fill_between(dates_te,  sp_full.iloc[idx_te],  alpha=0.45, color=COLORS['test'],  label=f'Test   ({n_te} obs)')
ax.axvline(dates_val[0], color=COLORS['val'],  lw=1.5, ls='--')
ax.axvline(dates_te[0],  color=COLORS['test'], lw=1.5, ls='--')
ax.set_title('S&P 500 Daily Log Returns — Chronological 60/20/20 Split', fontweight='bold')
ax.set_ylabel('Log return')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()


---
## Q1 — Overfitting vs. Underfitting  *(8 pts)*

Four scenarios are described below. For each one, state whether it is an example of
**overfitting**, **underfitting**, or **neither**, and give a one-sentence justification.

| # | Scenario | Classification | Justification |
|---|----------|----------------|---------------|
| A | An AR(15) model on daily stock returns has train RMSE = 0.0088 and val RMSE = 0.0118. The gap widens as p increases. | | |
| B | A naive forecast (tomorrow = today) produces nearly the same RMSE as a complex model on a highly persistent series. | | |
| C | A linear trend model applied to a strongly seasonal retail series has high train RMSE and high val RMSE, both similarly large. | | |
| D | An AR(2) model has in-sample R² = 0.91 but out-of-sample R² = −0.05. | | |


### Sample Answers — Q1

| Scenario | Label | Reason |
|---|---|---|
| A model memorises noise in training data, test RMSE is 3× train RMSE | **Overfitting** | High gap between train and test error is the signature of overfitting |
| AR(1) on a white-noise series; both train and test RMSE ≈ σ | **Neither** | A simple model on an inherently unpredictable series cannot overfit or underfit |
| A linear trend is fit to a clearly nonlinear seasonal series; both errors are high | **Underfitting** | Model is too simple to capture the true pattern; both train and test error are elevated |
| A polynomial of degree 20 fit to 25 training points; test RMSE explodes | **Overfitting** | Far too many parameters for the sample size; the model fits noise |


---
## Q2 — Bias–Variance Tradeoff  *(8 pts)*

Answer the following questions in 2–3 sentences each.

**1. In your own words, explain what bias and variance mean in the context of a forecasting model.**


**2. Why does increasing model complexity always reduce training error but not always reduce forecast error?**

**3. In high-stakes settings like credit approval or algorithmic trading, practitioners tend to be more worried about overfitting than underfitting.
Why might false confidence from an overfitting model be more costly than the modest gains from a slightly more complex model?**


---
## Q3 ⭐ — AR(p) Complexity Curve  *(14 pts — write your code below)*

**Task:** Fit AR(p) models for p = 1, 2, …, 15 using only the **training set**.
Evaluate each model on the **validation set**. Then produce a two-panel figure:
- **Left panel:** Train RMSE and Val RMSE vs. lag order p (both on the same axes).
- **Right panel:** Bar chart of the overfitting gap = Val RMSE − Train RMSE per p.
- Mark the lag order that minimises Val RMSE with a vertical dashed line on both panels.

**Hint:** To build AR(p) features, for each time t ≥ p, set `X[t] = [log_ret[t-1], ..., log_ret[t-p]]`.
Only use training observations when fitting — never let validation data enter the fit.


In [ ]:
# ── Q3: AR(p) Complexity Curve ─────────────────────────────────────────

max_p         = 15
tr_rmse_list  = []
val_rmse_list = []

for p in range(1, max_p + 1):
    X_p, y_p = [], []
    for t in range(p, n):
        X_p.append([log_ret[t - k] for k in range(1, p + 1)])
        y_p.append(log_ret[t])
    X_p = np.array(X_p)
    y_p = np.array(y_p)

    tr_end  = n_tr - p
    val_end = tr_end + n_val

    m = LinearRegression()
    m.fit(X_p[:tr_end], y_p[:tr_end])

    tr_rmse_list.append(rmse(y_p[:tr_end],       m.predict(X_p[:tr_end])))
    val_rmse_list.append(rmse(y_p[tr_end:val_end], m.predict(X_p[tr_end:val_end])))

orders       = np.arange(1, max_p + 1)
gap          = np.array(val_rmse_list) - np.array(tr_rmse_list)
best_p       = orders[np.argmin(val_rmse_list)]
val_rmse_arr = np.array(val_rmse_list)
tr_rmse_arr  = np.array(tr_rmse_list)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(orders, tr_rmse_arr,  'o-', color=COLORS['train'], label='Train RMSE',      lw=1.5)
ax.plot(orders, val_rmse_arr, 's-', color=COLORS['val'],   label='Validation RMSE', lw=1.5)
ax.axvline(best_p, color='red', ls='--', lw=1.2, label=f'Best p = {best_p}')
ax.set_xlabel('AR order p');  ax.set_ylabel('RMSE')
ax.set_title('AR(p) Complexity Curve');  ax.legend()

ax2 = axes[1]
bar_colors = [COLORS['red'] if g > 0 else COLORS['neutral'] for g in gap]
ax2.bar(orders, gap, color=bar_colors, edgecolor='white')
ax2.axhline(0, color='black', lw=0.8)
ax2.set_xlabel('AR order p');  ax2.set_ylabel('Val RMSE − Train RMSE')
ax2.set_title('Overfitting Gap (Val − Train RMSE)')

plt.tight_layout()
plt.show()
print(f'Best AR order (min Val RMSE): p = {best_p}')


---
## Q4 — Interpret the Complexity Curve  *(8 pts)*

Use your Q3 output to answer the questions below.

**1. At approximately which lag order p does overfitting begin?
What feature of the plot tells you this?**


**2. Training RMSE keeps falling as p increases. Why does this happen regardless of whether the added lags are informative?**

**3. The gap chart shows the Val RMSE − Train RMSE difference per lag order.
If this gap were negative for all p, what would that imply about your validation split?**


---
## Q5 — Time-Aware Validation Scenarios  *(8 pts)*

For each splitting approach below, state whether it is:
- ✅ **Correct** — respects temporal ordering
- ⚠️ **Look-ahead bias** — future data enters training implicitly
- ❌ **Data leakage** — future values appear directly as predictors

Then give a one-sentence explanation.

| # | Description | Verdict | Explanation |
|---|-------------|---------|-------------|
| A | You standardise the entire S&P 500 return series (mean and std computed over all 10 years), then split 60/20/20 chronologically. | | |
| B | You train on 2015–2018, validate on 2019–2020, and test on 2021–2024, fitting the scaler on training data only. | | |
| C | You use random K-fold (K=5) with shuffling to cross-validate an AR(3) model on monthly GDP growth. | | |
| D | You use next-day VIX change (observed after market close) to predict today's S&P 500 return before the open. | | |


### Sample Answers — Q5

| Approach | Status | Reason |
|---|---|---|
| Random 80/20 split on a time series | ⚠️ **Look-ahead bias** | Future observations enter the training set; model 'sees' the future |
| Chronological 60/20/20 split, test never touched during training | ✅ **Correct** | Respects temporal ordering; test set is a genuine holdout |
| 5-fold CV with folds randomly shuffled | ⚠️ **Look-ahead bias** | Same problem as random split — temporal structure is destroyed |
| Time-series split where each val fold always follows its training fold | ✅ **Correct** | Preserves causality; each step mimics real-time deployment |


---
## Q6 — Data Leakage: Identify and Explain  *(10 pts)*

The cell below fits two models and produces a scatter plot.
Run it, then answer the questions that follow.


In [ ]:
# ── Q6 Pre-built: Legitimate AR(1) vs. Leaky model ─────────────────────
# Do not modify — just run and observe the output.

# Legitimate model: predict log_ret[t] using log_ret[t-1]  (correct — lag 1)
X_leg = log_ret[:-1].reshape(-1, 1)   # y_{t-1}
y_leg = log_ret[1:]                    # y_t

# Leaky model: predict log_ret[t] using log_ret[t+1]  (future — not observable at t)
X_lk  = log_ret[1:].reshape(-1, 1)    # y_{t+1}  ← leaks the future!
y_lk  = log_ret[:-1]                  # y_t

split = int(len(y_leg) * 0.60)

m_leg = LinearRegression().fit(X_leg[:split], y_leg[:split])
m_lk  = LinearRegression().fit(X_lk[:split],  y_lk[:split])

r2_leg_tr = r2_score(y_leg[:split],  m_leg.predict(X_leg[:split]))
r2_lk_tr  = r2_score(y_lk[:split],   m_lk.predict(X_lk[:split]))
r2_leg_te = r2_score(y_leg[split:],  m_leg.predict(X_leg[split:]))
r2_lk_te  = r2_score(y_lk[split:],   m_lk.predict(X_lk[split:]))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, Xte, yte, model, label, r2tr, r2te, color in [
    (axes[0], X_leg[split:], y_leg[split:], m_leg,
     'Legitimate AR(1)',  r2_leg_tr, r2_leg_te, COLORS['train']),
    (axes[1], X_lk[split:],  y_lk[split:],  m_lk,
     'Leaky Model (uses future!)', r2_lk_tr, r2_lk_te, COLORS['red']),
]:
    yhat = model.predict(Xte)
    ax.scatter(yte, yhat, s=6, alpha=0.3, color=color)
    lims = [min(yte.min(), yhat.min()), max(yte.max(), yhat.max())]
    ax.plot(lims, lims, 'k--', lw=1)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Actual log return')
    ax.set_ylabel('Predicted log return')
    ax.annotate(f'Train R² = {r2tr:.4f}\nTest  R² = {r2te:.4f}',
                xy=(0.05, 0.88), xycoords='axes fraction', fontsize=10,
                bbox=dict(boxstyle='round', fc='white', alpha=0.8))

fig.suptitle('Q6 — Legitimate vs. Leaky Forecast Model', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()


**1. The leaky model has a near-perfect R² on both train and test. Explain in two sentences
why this is a false signal rather than a genuine forecasting achievement.**


**2. The legitimate AR(1) has a very low test R². Does this mean the model is bad? Or does it reflect
something true about the data-generating process? Explain.**


**3. Name one realistic example of data leakage that would be harder to spot than the one above —
something a practitioner might accidentally introduce without realising.**


---
## Q7 ⭐ — Rolling vs. Expanding Window Validation  *(14 pts — write your code below)*

**Task:** Using the S&P 500 log return series, implement two one-step-ahead AR(1)
forecasting schemes over the **validation period only** (indices `n_tr` to `n_tr + n_val`):

1. **Rolling window** — at each step t, train on the most recent `W = 252` observations before t.
2. **Expanding window** — at each step t, train on all observations from the start up to t.

For each method, collect the one-step-ahead forecast errors.
Then produce a **single plot** showing the cumulative RMSE of both methods over the validation period.

**Hint:** For AR(1), your feature at step t is `[log_ret[t-1]]` and your target is `log_ret[t]`.
The rolling window uses `log_ret[t-W:t]` to build the training set.
The expanding window uses `log_ret[0:t]` (all available history up to t).


In [ ]:
# ── Q7: Rolling vs. Expanding AR(1) ────────────────────────────────────
W = 252    # rolling window size (≈ 1 trading year)

roll_errors = []
exp_errors  = []
val_dates   = []

for t in range(n_tr, n_tr + n_val):

    # ── Rolling window ───────────────────────────────────────────────────
    train_roll = log_ret[t - W : t]
    X_roll = train_roll[:-1].reshape(-1, 1)
    y_roll = train_roll[1:]
    m_roll = LinearRegression().fit(X_roll, y_roll)
    pred_roll = m_roll.predict([[log_ret[t - 1]]])[0]
    roll_errors.append(log_ret[t] - pred_roll)

    # ── Expanding window ─────────────────────────────────────────────────
    train_exp = log_ret[: t]
    X_exp = train_exp[:-1].reshape(-1, 1)
    y_exp = train_exp[1:]
    m_exp = LinearRegression().fit(X_exp, y_exp)
    pred_exp = m_exp.predict([[log_ret[t - 1]]])[0]
    exp_errors.append(log_ret[t] - pred_exp)

    val_dates.append(dates_r[t])

roll_errors = np.array(roll_errors)
exp_errors  = np.array(exp_errors)

# ── Cumulative RMSE ───────────────────────────────────────────────────────
cum_rmse_roll = [np.sqrt(np.mean(roll_errors[:i+1]**2)) for i in range(len(roll_errors))]
cum_rmse_exp  = [np.sqrt(np.mean(exp_errors[:i+1]**2))  for i in range(len(exp_errors))]

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(val_dates, cum_rmse_roll, color=COLORS['val'],   lw=1.5, label='Rolling  AR(1)')
ax.plot(val_dates, cum_rmse_exp,  color=COLORS['train'], lw=1.5, label='Expanding AR(1)')
ax.set_xlabel('Date');  ax.set_ylabel('Cumulative RMSE')
ax.set_title('Rolling vs. Expanding AR(1) — Cumulative Val RMSE');  ax.legend()
plt.tight_layout();  plt.show()

print(f'Rolling  AR(1)  Val RMSE: {rmse(roll_errors, np.zeros_like(roll_errors)):.6f}')
print(f'Expanding AR(1) Val RMSE: {rmse(exp_errors, np.zeros_like(exp_errors)):.6f}')


---
## Q8 — Interpret Rolling vs. Expanding Results  *(10 pts)*

Use your Q7 output to answer the questions below.

**1. Which method performs better overall on the validation set — rolling or expanding?
Does one method clearly dominate, or do they trade off at different periods?**


**2. A retail forecasting team is predicting weekly store demand. Their data shows strong seasonal
patterns that shift every year due to changing consumer habits and new product launches.
Would you recommend a rolling or expanding window for this setting? Justify your choice.**


**3. Suppose you shorten the rolling window from W=252 to W=60 days.
What would you expect to happen to the RMSE curve? What is the tradeoff?**


---
## Q9 — Why Forecasts Fail: Regime Change  *(10 pts)*

The cell below trains an AR(2) model on the pre-COVID training set and plots rolling RMSE.
Run it, then answer the questions.


In [ ]:
# ── Q9 Pre-built: rolling RMSE with COVID shock ─────────────────────────
# Do not modify — just run.

p = 2
X_ar2, y_ar2 = [], []
for t in range(p, n):
    X_ar2.append([log_ret[t-1], log_ret[t-2]])
    y_ar2.append(log_ret[t])
X_ar2 = np.array(X_ar2); y_ar2 = np.array(y_ar2)
dates_ar2 = dates_r[p:]

m_ar2 = LinearRegression().fit(X_ar2[:n_tr-p], y_ar2[:n_tr-p])
pred_ar2 = m_ar2.predict(X_ar2)
errs_ar2 = y_ar2 - pred_ar2

# Rolling 60-day RMSE
roll_w = 60
roll_rmse_vals = []
roll_rmse_dates = []
for i in range(roll_w, len(errs_ar2)):
    roll_rmse_vals.append(np.sqrt(np.mean(errs_ar2[i-roll_w:i]**2)))
    roll_rmse_dates.append(dates_ar2[i])

roll_rmse_vals  = np.array(roll_rmse_vals)
roll_rmse_dates = pd.DatetimeIndex(roll_rmse_dates)

# Approximate COVID crash window
covid_start = pd.Timestamp('2020-02-15')
covid_end   = pd.Timestamp('2020-05-01')

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                          gridspec_kw={'height_ratios': [1.8, 1.2]})

ax = axes[0]
sp_s = np.log(df['SP500']).diff().dropna().reindex(dates_ar2)
ax.plot(dates_ar2, sp_s, color=COLORS['neutral'], lw=0.8, alpha=0.7, label='S&P 500 log return')
ax.axvspan(covid_start, covid_end, alpha=0.25, color='red', label='COVID crash window')
ax.axvline(dates_tr[-1], color=COLORS['train'], ls='--', lw=1.2, label='Train / Val boundary')
ax.set_ylabel('Log return')
ax.legend(fontsize=9)
ax.set_title('Q9 — AR(2) Rolling RMSE and Regime Change', fontweight='bold')

ax2 = axes[1]
ax2.plot(roll_rmse_dates, roll_rmse_vals, color=COLORS['accent'], lw=1.5, label='60-day rolling RMSE')
ax2.axvspan(covid_start, covid_end, alpha=0.25, color='red')
ax2.axvline(dates_tr[-1], color=COLORS['train'], ls='--', lw=1.2)
ax2.set_ylabel('RMSE (60-day rolling)')
ax2.set_xlabel('Date')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()


**1. Look at the rolling RMSE chart. Describe what happens to RMSE around the COVID crash window. Which of the three failure modes (regime change, parameter instability, extrapolation risk) best explains this pattern?**

**2. The AR(2) model was trained entirely before COVID. How would you test whether the model's coefficients have become unreliable after the shock?**

**3. After the COVID window, rolling RMSE gradually returns toward its pre-shock level.
What does this suggest about the data-generating process — did a permanent structural break occur,
or was this a temporary volatility spike? How would you distinguish the two?**


---
## Q10 — Block Bootstrap & Forecast Uncertainty  *(10 pts)*

The cell below generates block-bootstrap prediction intervals for an AR(2) model
at horizons h = 1, 5, 10, and 20 days. Run it, then answer the questions.


In [ ]:
# ── Q10 Pre-built: block bootstrap fan chart ────────────────────────────
# Do not modify — just run.

np.random.seed(99)
B      = 400    # bootstrap replications
block  = 20     # block length
H      = 20     # max horizon

# Fit AR(2) on full training set
m_boot = LinearRegression().fit(X_ar2[:n_tr-2], y_ar2[:n_tr-2])
resids = y_ar2[:n_tr-2] - m_boot.predict(X_ar2[:n_tr-2])
phi0, phi1, phi2 = m_boot.intercept_, m_boot.coef_[0], m_boot.coef_[1]

# Block bootstrap
n_blocks = len(resids) // block
fc_boot  = np.zeros((B, H))
y_last   = log_ret[n_tr-1]
y_last2  = log_ret[n_tr-2]

for b in range(B):
    block_starts = np.random.randint(0, len(resids)-block, n_blocks)
    boot_resid   = np.concatenate([resids[s:s+block] for s in block_starts])
    y1, y2 = y_last, y_last2
    for h in range(H):
        e         = boot_resid[h % len(boot_resid)]
        y_next    = phi0 + phi1*y1 + phi2*y2 + e
        fc_boot[b, h] = y_next
        y2, y1 = y1, y_next

horizons  = np.arange(1, H+1)
pt_fc     = [phi0 + phi1*y_last + phi2*y_last2] + [phi0]*(H-1)  # approx
p10, p90  = np.percentile(fc_boot, 10, axis=0), np.percentile(fc_boot, 90, axis=0)
p25, p75  = np.percentile(fc_boot, 25, axis=0), np.percentile(fc_boot, 75, axis=0)

fig, ax = plt.subplots(figsize=(11, 5))
ax.fill_between(horizons, p10, p90, alpha=0.25, color=COLORS['purple'], label='80% interval')
ax.fill_between(horizons, p25, p75, alpha=0.40, color=COLORS['purple'], label='50% interval')
ax.plot(horizons, pt_fc, color=COLORS['accent'], lw=2.5, label='Point forecast')
ax.axhline(0, color='black', lw=0.8, ls='--', alpha=0.4)
ax.set_xlabel('Forecast horizon h (trading days)', fontsize=11)
ax.set_ylabel('Predicted log return', fontsize=11)
ax.set_title('Q10 — AR(2) Block Bootstrap Prediction Intervals\n'
             f'B={B} replications, block length={block}', fontweight='bold', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f'80% interval at h=1 : [{p10[0]*100:.2f}%, {p90[0]*100:.2f}%]')
print(f'80% interval at h=10: [{p10[9]*100:.2f}%, {p90[9]*100:.2f}%]')
print(f'80% interval at h=20: [{p10[19]*100:.2f}%, {p90[19]*100:.2f}%]')


**1. The fan chart shows prediction intervals widening as the horizon h increases. Explain in 2 sentences why this happens, relating your answer to the concept of compounding forecast uncertainty.**

**2. The 80% interval at h=20 is printed above. Suppose a portfolio manager holds a position for
20 trading days. Describe in plain English what the 80% interval means for their risk assessment.**


**3. Why is a block bootstrap used instead of a standard bootstrap when resampling time-series residuals?
What would go wrong if you resampled individual residuals independently?**


---
## Instructor Notes

This is the **instructor version** with sample answers for all written questions and complete code for Q3 and Q7.

When grading student submissions, look for:
- [ ] Q1: Correct identification of overfitting vs. underfitting with valid reasoning.
- [ ] Q2: Clear explanation of bias/variance with the tradeoff intuition.
- [ ] Q3 ⭐: Correct AR(p) loop with chronological split; complexity curve shows U-shape in val RMSE.
- [ ] Q4: Student references their own Q3 plot output.
- [ ] Q5: Correct identification of look-ahead bias scenarios.
- [ ] Q6: Student correctly identifies future leakage as the cause of inflated R².
- [ ] Q7 ⭐: Rolling and expanding windows correctly implemented; cumulative RMSE plot shown.
- [ ] Q8–Q10: Written reasoning is specific and connects to the pre-built plots.
